In [ ]:
!pip -q install unsloth transformers datasets trl peft accelerate bitsandbytes sentencepiece google-generativeai


In [ ]:
from pathlib import Path
import json
from datasets import Dataset

data_path = Path("data/train.jsonl")
if not data_path.exists():
    raise FileNotFoundError(f"Missing dataset: {data_path}")

records = []
with data_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        records.append(json.loads(line))

print("records:", len(records))
print("sample keys:", list(records[0].keys()))


In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported, add_new_tokens

max_seq_length = 2048
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

add_new_tokens(model, tokenizer, new_tokens=["<think>", "</think>"])

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
    use_rslora=False,
    loftq_config=None,
)


In [ ]:
import matplotlib.pyplot as plt

def normalize_messages(record):
    messages = record.get("messages") or record.get("conversations") or record.get("prompt")
    if isinstance(messages, list):
        normalized = []
        for msg in messages:
            role = str(msg.get("role") or msg.get("from") or "user").lower()
            content = str(msg.get("content") or msg.get("value") or "")
            if not content:
                continue
            if role in {"human", "user"}:
                role = "user"
            elif role in {"assistant", "gpt", "bot"}:
                role = "assistant"
            elif role != "system":
                role = "user"
            normalized.append({"role": role, "content": content})
        return normalized
    user_text = record.get("input") or record.get("question") or record.get("prompt") or ""
    answer_text = record.get("output") or record.get("answer") or ""
    return [
        {"role": "system", "content": "You are vAGI, created by Vietrix."},
        {"role": "user", "content": str(user_text)},
        {"role": "assistant", "content": str(answer_text)},
    ]

def render_text(messages):
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return "\n".join(f"{m['role']}: {m['content']}" for m in messages)

texts = [render_text(normalize_messages(r)) for r in records]
dataset = Dataset.from_dict({"text": texts})

lengths = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in texts]
plt.hist(lengths, bins=30)
plt.title("Token length distribution")
plt.xlabel("tokens")
plt.ylabel("count")
plt.show()


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    output_dir="outputs/vagi_lab",
    optim="adamw_8bit",
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

trainer.train()


In [ ]:
FastLanguageModel.for_inference(model)

prompt = [
    {"role": "system", "content": "You are vAGI, created by Vietrix."},
    {"role": "user", "content": "Who are you?"},
]

device = "cuda" if torch.cuda.is_available() else "cpu"
input_ids = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    return_tensors="pt",
).to(device)

outputs = model.generate(input_ids=input_ids, max_new_tokens=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
